In [5]:
clear all

In [6]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"C:\Users\41240\OneDrive\桌面\MBS\2026 fall\Advanced Analytics\data"
)

OD_FILE = DATA_DIR / "nj_od_main_JT00_2021.csv.gz"
XWALK_FILE = DATA_DIR / "nj_xwalk.csv.gz"

# 先读取通勤文件前 5 行，检查格式
# 编号按文本读取，保留完整编号和前导零
od_preview = pd.read_csv(
    OD_FILE,
    usecols=["h_geocode", "w_geocode", "S000"],
    dtype={"h_geocode": "string", "w_geocode": "string"},
    nrows=5
)

# 地理对应表只读取需要的两列
xwalk = pd.read_csv(
    XWALK_FILE,
    usecols=["tabblk2020", "zcta"],
    dtype="string"
)

print("通勤文件样例：")
display(od_preview)

print("Block → ZCTA 对应表样例：")
display(xwalk.head())

print("对应表行数：", len(xwalk))
print("重复 Block 编号：", xwalk["tabblk2020"].duplicated().sum())
print("缺失 ZCTA：", xwalk["zcta"].isna().sum())

通勤文件样例：


,w_geocode,h_geocode,S000
0,340010001001002,340010001001003,1
1,340010001001002,340010001001009,1
2,340010001001002,340010001001015,1
3,340010001001002,340010001002009,1
4,340010001001002,340010024001009,1


Block → ZCTA 对应表样例：


,tabblk2020,zcta
0,340010001001000,08401
1,340010001001001,08401
2,340010001001002,08401
3,340010001001003,08401
4,340010001001004,08401


对应表行数： 137972
重复 Block 编号： 0
缺失 ZCTA： 0


In [7]:
# 建立 Block → ZCTA 查询表
block_to_zcta = xwalk.set_index("tabblk2020")["zcta"]

parts = []
total_rows = 0
total_jobs = 0

# 分批读取完整文件，减少内存占用
for chunk in pd.read_csv(
    OD_FILE,
    usecols=["h_geocode", "w_geocode", "S000"],
    dtype={"h_geocode": "string", "w_geocode": "string"},
    chunksize=250_000
):
    # 居住地 → 工作地
    chunk["origin_zcta"] = chunk["h_geocode"].map(block_to_zcta)
    chunk["destination_zcta"] = chunk["w_geocode"].map(block_to_zcta)

    # 检查匹配和岗位数量，发现问题就停止
    for column in ["origin_zcta", "destination_zcta"]:
        valid = (
            chunk[column].str.fullmatch(r"[0-9]{5}").fillna(False)
            & ~chunk[column].isin(["00000", "99999"])
        )
        assert valid.all(), f"{column} 存在未匹配或无效地区"

    assert chunk["S000"].notna().all(), "存在缺失岗位数量"
    assert chunk["S000"].ge(0).all(), "存在负数岗位数量"

    total_rows += len(chunk)
    total_jobs += int(chunk["S000"].sum())

    # 每一批先汇总
    grouped = chunk.groupby(
        ["origin_zcta", "destination_zcta"],
        as_index=False
    )["S000"].sum()

    parts.append(grouped)

# 合并各批次，再汇总同一对地区
commute_flows = (
    pd.concat(parts, ignore_index=True)
    .groupby(["origin_zcta", "destination_zcta"], as_index=False)["S000"]
    .sum()
    .rename(columns={"S000": "commute_jobs"})
)

# 确认汇总前后岗位总数一致
assert commute_flows["commute_jobs"].sum() == total_jobs

# 保存到当前 Notebook 所在的项目文件夹
output_dir = Path.cwd() / "commute_outputs"
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "nj_commute_zcta_flows_2021.csv"
commute_flows.to_csv(output_file, index=False, encoding="utf-8-sig")

print("处理的原始行数：", total_rows)
print("汇总后的地区对数：", len(commute_flows))
print("岗位总数：", total_jobs)
print("保存位置：", output_file.resolve())

display(commute_flows.head(10))

处理的原始行数： 3222070
汇总后的地区对数： 181994
岗位总数： 3439824
保存位置： C:\Users\41240\OneDrive\文档\GitHub\practicum-project\commute_outputs\nj_commute_zcta_flows_2021.csv


,origin_zcta,destination_zcta,commute_jobs
0,07001,07001,250
1,07001,07002,27
2,07001,07003,17
3,07001,07004,10
4,07001,07005,2
5,07001,07006,3
6,07001,07008,123
7,07001,07009,3
8,07001,07011,4
9,07001,07012,9


In [8]:
# 区分同一地区内部和跨地区联系
flows = commute_flows.copy()
flows["is_external"] = (
    flows["origin_zcta"] != flows["destination_zcta"]
)

# 所有出现在通勤表中的地区
regions = pd.Index(
    sorted(
        set(flows["origin_zcta"])
        | set(flows["destination_zcta"])
    ),
    name="zcta"
)

features = pd.DataFrame(index=regions)

# 按居住地汇总：该地区居民对应的岗位总数
features["resident_jobs_total"] = (
    flows.groupby("origin_zcta")["commute_jobs"].sum()
)

# 按工作地汇总：位于该地区的岗位总数
features["workplace_jobs_total"] = (
    flows.groupby("destination_zcta")["commute_jobs"].sum()
)

external = flows.loc[flows["is_external"]]
internal = flows.loc[~flows["is_external"]]

# 其他地区 → 本地区
features["inbound_jobs_external"] = (
    external.groupby("destination_zcta")["commute_jobs"].sum()
)

# 本地区 → 其他地区
features["outbound_jobs_external"] = (
    external.groupby("origin_zcta")["commute_jobs"].sum()
)

# 居住地和工作地都在本地区
features["internal_jobs"] = (
    internal.groupby("origin_zcta")["commute_jobs"].sum()
)

# 完整通勤表中没有对应联系的数量记为 0
features = features.fillna(0).astype("int64")

# 分母为 0 时，占比保持缺失
features["external_inbound_share"] = (
    features["inbound_jobs_external"]
    / features["workplace_jobs_total"].where(
        features["workplace_jobs_total"] > 0
    )
)

features["external_outbound_share"] = (
    features["outbound_jobs_external"]
    / features["resident_jobs_total"].where(
        features["resident_jobs_total"] > 0
    )
)

# 核对：内部 + 外部 = 总数
assert (
    features["internal_jobs"]
    + features["inbound_jobs_external"]
).eq(features["workplace_jobs_total"]).all()

assert (
    features["internal_jobs"]
    + features["outbound_jobs_external"]
).eq(features["resident_jobs_total"]).all()

features = features.reset_index()

features.to_csv(
    output_dir / "nj_commute_zcta_features_2021.csv",
    index=False,
    encoding="utf-8-sig"
)

display(features.head(10))

,zcta,resident_jobs_total,workplace_jobs_total,inbound_jobs_external,outbound_jobs_external,internal_jobs,external_inbound_share,external_outbound_share
0,07001,6350,10337,10087,6100,250,0.975815,0.960630
1,07002,22969,12630,7991,18330,4639,0.632700,0.798032
2,07003,20829,12403,10853,19279,1550,0.875030,0.925585
3,07004,3474,19552,19155,3077,397,0.979695,0.885723
4,07005,7484,4490,3968,6962,522,0.883742,0.930251
5,07006,11071,10192,9062,9941,1130,0.889129,0.897932
6,07008,9454,9470,8518,8502,952,0.899472,0.899302
7,07009,5399,4060,3711,5050,349,0.914039,0.935358
8,07010,6260,2179,1586,5667,593,0.727857,0.905272
9,07011,14578,7868,7168,13878,700,0.911032,0.951982
